In [1]:
import os, warnings
warnings.filterwarnings('ignore')
import pertpy as pt
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
ad_hepg2 = sc.read_h5ad("raw/NadigOConner2024_hepg2.h5ad"); print(ad_hepg2)
ad_jurkat = sc.read_h5ad("raw/NadigOConner2024_jurkat.h5ad"); print(ad_jurkat)

AnnData object with n_obs × n_vars = 145473 × 9624
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'disease', 'cancer', 'cell_line', 'organism', 'perturbation_type', 'tissue_type', 'perturbation', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'


AnnData object with n_obs × n_vars = 262956 × 8882
    obs: 'batch', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'disease', 'cancer', 'cell_line', 'organism', 'perturbation_type', 'tissue_type', 'perturbation', 'ncounts', 'ngenes', 'nperts', 'percent_ribo'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells'


In [3]:
padata_dict = {
    'hepg2': ad_hepg2,
    'jurkat': ad_jurkat,
}

processed_padata_dict = {}
for ct, pad_ct in padata_dict.items():
    print(f"Processing {ct}")
    pad = pad_ct.copy()
    pad.X = pad.X.astype(np.float32)
    sc.pp.filter_cells(pad, min_genes=200)
    sc.pp.filter_genes(pad, min_cells=50)
    num_cell_per_pert = pad.obs.groupby('perturbation').size()
    included_perts = num_cell_per_pert[lambda x: x > 50].index
    pad = pad[pad.obs['perturbation'].isin(included_perts)]
    pad.layers["raw"] = pad.X.copy()
    sc.pp.normalize_total(pad, target_sum=1e4)
    sc.pp.log1p(pad)
    ms = pt.tl.Mixscape()
    ms.perturbation_signature(
        pad,
        pert_key="perturbation",
        control="control",
    )
    ms.mixscape(
        pad,
        labels="perturbation",
        control="control",
        layer='X_pert'
    )
    print(f"filter {(pad.obs['mixscape_class_global']=='NP').sum()} NP cells")
    pad = pad[pad.obs['mixscape_class_global'] != 'NP']
    processed_padata_dict[ct] = pad

Processing hepg2


filter 43178 NP cells
Processing jurkat


filter 123105 NP cells


In [4]:
pertgenes_sel = np.intersect1d(*[
    pad.obs["perturbation"].value_counts()[lambda x: x > 50].index
    for ct, pad in processed_padata_dict.items()
])

In [ ]:
padata = sc.concat([pad[pad.obs['perturbation'].isin(pertgenes_sel)] for ct, pad in processed_padata_dict.items()])
sc.pp.highly_variable_genes(padata, n_top_genes=5000)

padata.write_h5ad("preprocessed.h5ad")

### split dataset

In [6]:
np.random.seed(42)
split_df = (
    padata
    .obs[['cell_line']]
    .copy()
    .reset_index(names='cell')
    .rename(columns={'cell_line': 'subsplit'})
)
split_df['presplit'] = np.random.choice(
    ['train', 'val', 'test'], 
    size=split_df.shape[0], 
    p=[0.7, 0.1, 0.2], 
    replace=True
)


In [ ]:
# train on Hep-G2 and test on Jurkat
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'Hep-G2' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonhepg2.csv', index=False)
pd.read_csv('split_trainonhepg2.csv').groupby(['split', 'subsplit']).size()

split  subsplit
test   Hep-G2       7540
       Jurkat      64072
train  Hep-G2      26687
val    Hep-G2       3733
dtype: int64

In [ ]:
# train on Jurkat and test on Hep-G2
split_df['split'] = split_df[['subsplit', 'presplit']].apply(
    lambda x: x['presplit'] if x['subsplit'] == 'Jurkat' else 'test', axis=1
)
split_df[['cell', 'split', 'subsplit']].to_csv('split_trainonjurkat.csv', index=False)
pd.read_csv('split_trainonjurkat.csv').groupby(['split', 'subsplit']).size()

split  subsplit
test   Hep-G2      37960
       Jurkat      12779
train  Jurkat      44851
val    Jurkat       6442
dtype: int64